In [0]:
#Step1
#create target data frame from existing scd type2 table without scd type2 meta data columns
emp_target_df=spark.read.format("delta").table("demo_catalog.demo_schema.emp_target_scdtype2")

#then drop scd type2 metadata column fron the dataframe
emp_target_df= emp_target_df.drop("start_date", "end_date", "is_active")
emp_target_df.display()


In [0]:
#Step 2 create source data frame- from adls 
emp_source=[(4, "Ramesh", "IT", "Mumbai"),
            (5, "Sunil", "FA", "Pune")
                        ]

#create source dataframe
emp_source_df= spark.createDataFrame(emp_source, ["id", "name", "department", "city"])
emp_source_df.show()

In [0]:
#step3 generate hash key for source and target dataframe
from pyspark.sql.functions import xxhash64,col, monotonically_increasing_id
emp_target_df= emp_target_df.withColumn("hash_key", xxhash64(col("id"), col("name"),col("department"), col("city")))
emp_target_df.display()

emp_source_df= emp_source_df.withColumn("hash_key", xxhash64(col("id"), col("name"),col("department"), col("city")))
emp_source_df.display()


In [0]:
#Step4. Perform left anti join on source and target which filter out only matching from left table
#means new records 
result_df= emp_source_df.join(emp_target_df, on="hash_key", how="left_anti").drop("hash_key")
result_df.display()

In [0]:
#Step5. Add scd type 2 columns in resulted df 
from pyspark.sql.functions import lit, current_timestamp, to_timestamp
result_df= result_df.withColumn("start_date", to_timestamp(current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("end_date", to_timestamp(lit("9999-12-31 00:00:00"), "yyyy-MM-dd HH:mm:ss")) \
        .withColumn("is_active", lit("Y"))
result_df.display()

In [0]:
#Step6. Append the resulted df to target table
result_df.write.mode("append").saveAsTable("demo_catalog.demo_schema.emp_target_scdtype2")

In [0]:
#step7 Create a new target dataframe from scd type 2 table
# Then use row_number() window function partition by emp_id and order by start date desc to assign rank to each partition by employee id

#create a df from emp_target_scdtype2 table
target_df= spark.read.format("delta").table("demo_catalog.demo_schema.emp_target_scdtype2")

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
windowSpec= Window.partitionBy("id").orderBy((target_df.start_date).desc())
rank_target_df= target_df.withColumn("rank", row_number().over(windowSpec))
rank_target_df.display()

In [0]:
#Step8- We will perform two action. we have to update end_date and is_active flag 
# We will update the end_date column value for each row and end_date value will take previous row start date for that partition using lag window function
#Then change the is_active status to N where end_date is not equal to 9999-12-31 00:00:00 or rank greater than 1

from pyspark.sql.functions import lag, when, col
from pyspark.sql.window import Window

#create window spec as we will use lag
windowSpec= Window.partitionBy("id").orderBy(rank_target_df.rank)

#update end_date by using lag() to fetch previosur row start date in the condition
update_enddate_df= rank_target_df.withColumn("end_date", when((rank_target_df.rank) >1, lag("start_date",1).over(windowSpec)).otherwise(rank_target_df.end_date)
                                            )
#update is_active based on the new end_date
final_df= update_enddate_df.withColumn("is_active", when((update_enddate_df.rank) > 1, "N").otherwise(update_enddate_df.is_active) 
                                       )
#drop rank column
final_df=final_df.drop("rank")
#display result
final_df.display()

In [0]:
#Step9. Finally write the final_df output to scd type 2 table and we are done.
final_df.write.mode("overwrite").saveAsTable("demo_catalog.demo_schema.emp_target_scdtype2")

In [0]:
%sql
---query the final dimention table or scd type 2 table
select * from demo_catalog.demo_schema.emp_target_scdtype2;